
# SageMaker XGBoost (Wine) — End‑to‑End
**Includes:** S3 upload helper • configurable instance types • real‑time inference • **Batch Transform** • offline **metrics (AUC/accuracy/F1/confusion matrix)** • cleanup.


## 0) Parameters

In [ ]:

import os, io, csv, json, boto3, sagemaker
import pandas as pd
from sagemaker import image_uris, get_execution_role
from sagemaker.session import Session

# ---- Edit these as needed ----
bucket = "<YOUR-S3-BUCKET-NAME>"       # REQUIRED: change to your S3 bucket
prefix = "wine-xgb-demo"               # S3 prefix/folder
train_instance_type = "ml.c5.2xlarge"  # training instance
deploy_instance_type = "ml.m4.xlarge"  # endpoint instance
use_spot_instances = True              # save cost on training
max_run_seconds = 1800                 # 30 min max
# --------------------------------

session = boto3.session.Session()
region = session.region_name
sm_session = sagemaker.Session()
role = get_execution_role()

print(f"Region: {region}\nRole: {role}")
if bucket.startswith("<"):
    raise ValueError("Please set 'bucket' to your S3 bucket name before running the next cells.")


## 1) Local CSVs → S3 Upload

In [ ]:

# Expect these files to be present locally (provided with this notebook)
local_train = "/mnt/data/wine_train_four.csv"
local_test  = "/mnt/data/wine_test_four.csv"
for p in (local_train, local_test):
    if not os.path.exists(p):
        raise FileNotFoundError(f"Missing local file: {p}")
print("Found local CSVs:", local_train, local_test)

# Upload to S3
s3 = session.resource("s3")
train_key = f"{prefix}/train/wine_train_four.csv"
test_key  = f"{prefix}/test/wine_test_four.csv"
s3.Bucket(bucket).upload_file(local_train, train_key)
s3.Bucket(bucket).upload_file(local_test,  test_key)

s3_train_uri = f"s3://{bucket}/{train_key}"
s3_test_uri  = f"s3://{bucket}/{test_key}"
print("Uploaded:")
print("  train ->", s3_train_uri)
print("  test  ->", s3_test_uri)


## 2) Training Inputs

In [ ]:

train_input = sagemaker.inputs.TrainingInput(s3_data=s3_train_uri, content_type="csv")
validation_input = sagemaker.inputs.TrainingInput(s3_data=s3_test_uri, content_type="csv")


## 3) Hyperparameters

In [ ]:

hyperparameters = {
    "max_depth": "5",
    "eta": "0.2",
    "gamma": "4",
    "min_child_weight": "6",
    "subsample": "0.7",
    "objective": "binary:logistic",  # label must be first column in CSV
    "num_round": 50
}
hyperparameters


## 4) Train with built‑in XGBoost

In [ ]:

container = image_uris.retrieve("xgboost", region=region, version="latest")
estimator = sagemaker.estimator.Estimator(
    image_uri=container,
    role=role,
    instance_count=1,
    instance_type=train_instance_type,
    output_path=f"s3://{bucket}/{prefix}/output",
    use_spot_instances=use_spot_instances,
    max_run=max_run_seconds,
    volume_size=5,
    hyperparameters=hyperparameters,
)
estimator.fit({"train": train_input, "validation": validation_input})


## 5) Deploy (real‑time endpoint)

In [ ]:

predictor = estimator.deploy(initial_instance_count=1, instance_type=deploy_instance_type)
from sagemaker.serializers import CSVSerializer
predictor.serializer = CSVSerializer()
predictor


## 6) Offline Metrics via Real‑Time Endpoint

In [ ]:

# Load local test set and compute metrics by calling the endpoint.
# Test CSV has label as first column; we send only features to the endpoint.
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix

test_df = pd.read_csv(local_test)
y_true = test_df.iloc[:, 0].values
X = test_df.iloc[:, 1:].values

# Batch predictions (avoid very large single payloads)
def predict_in_batches(X, batch_size=128):
    preds = []
    for i in range(0, len(X), batch_size):
        batch = X[i:i+batch_size]
        # predictor expects a list or CSV string; list -> serializer converts row-wise
        for row in batch:
            pred = predictor.predict(list(row)).decode("utf-8").strip()
            # Built-in XGBoost returns probability for positive class as a single float per line
            try:
                preds.append(float(pred))
            except:
                # Some containers may return trailing newlines; handle gracefully
                preds.extend([float(x) for x in pred.split() if x])
    return np.array(preds)

y_score = predict_in_batches(X, batch_size=64)
y_pred = (y_score >= 0.5).astype(int)

metrics = {
    "auc": float(roc_auc_score(y_true, y_score)),
    "accuracy": float(accuracy_score(y_true, y_pred)),
    "f1": float(f1_score(y_true, y_pred)),
    "confusion_matrix": confusion_matrix(y_true, y_pred).tolist()
}
metrics


## 7) Batch Transform (asynchronous batch inference)

In [ ]:

# For Batch Transform, we need a test file WITHOUT labels (features only).
features_only_path = "/tmp/wine_test_features_only.csv"
pd.DataFrame(X).to_csv(features_only_path, index=False, header=False)

# Upload features-only file
features_key = f"{prefix}/batch/features_only.csv"
s3.Bucket(bucket).upload_file(features_only_path, features_key)
s3_features_uri = f"s3://{bucket}/{features_key}"

# Create transformer from the trained estimator
transformer = estimator.transformer(
    instance_count=1,
    instance_type="ml.m5.large",
    assemble_with="Line",
    output_path=f"s3://{bucket}/{prefix}/batch-output"
)

# Run transform job
transformer.transform(
    data=s3_features_uri,
    content_type="text/csv",
    split_type="Line"
)
transformer.wait()
print("Batch output at:", transformer.output_path)


## 8) Evaluate Batch Transform Output

In [ ]:

# Download the single output file produced by batch transform and compute metrics
import re
s3_client = session.client("s3")

# List objects under the transform output prefix and find the .out file
out_prefix = f"{prefix}/batch-output"
resp = s3_client.list_objects_v2(Bucket=bucket, Prefix=out_prefix)
keys = [obj["Key"] for obj in resp.get("Contents", []) if obj["Key"].endswith(".out")]
if not keys:
    raise RuntimeError("No .out file found under batch output prefix.")

out_key = keys[0]
tmp_out = "/tmp/batch_predictions.out"
s3_client.download_file(bucket, out_key, tmp_out)

# Read predictions and compute metrics
with open(tmp_out, "r") as f:
    lines = [line.strip() for line in f if line.strip()]
y_score_bt = pd.Series(lines).astype(float).values
y_pred_bt = (y_score_bt >= 0.5).astype(int)

metrics_bt = {
    "auc": float(roc_auc_score(y_true, y_score_bt)),
    "accuracy": float(accuracy_score(y_true, y_pred_bt)),
    "f1": float(f1_score(y_true, y_pred_bt)),
    "confusion_matrix": confusion_matrix(y_true, y_pred_bt).tolist()
}
metrics_bt


## 9) Cleanup Resources

In [ ]:

# Delete real-time endpoint & model
try:
    predictor.delete_model()
    predictor.delete_endpoint()
    print("Deleted real-time endpoint & model.")
except Exception as e:
    print("Cleanup warning (endpoint/model):", e)

print("Note: Batch Transform jobs don't create persistent endpoints.")
